In [ ]:
import os
os.environ["OPENAI_API_KEY"]="YOUR_OPENAI_API_KEY"  # Replace with your actual OpenAI API key

In [5]:
!pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia langchain-chroma

**WIKIPEDIA** **RETRIEVER**

In [6]:
from langchain_community.retrievers import WikipediaRetriever

In [7]:
# Initialize the retriever (optional: set language and top_k)
ret = WikipediaRetriever(top_k_results=2, lang="en") #"When I give you a query, return at most 2 relevant Wikipedia results/documents." "This specifies the Wikipedia language edition to search."

In [9]:
query = "the geopolitical history of india and pakistan from the perspective of a chinese"
#get relevant wiki documents
docs = ret.invoke(query) #invoke so ret object is a runnable object

In [10]:
docs

[Document(metadata={'title': 'India–Pakistan war of 1971', 'summary': "The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.\nThirteen days after the war started, India achieved a clear advantage, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as the

In [14]:
for i, x in enumerate(docs):
  print(f"----- result {i+1}: -----")
  print(f"Content:\n{x.page_content}")

----- result 1: -----
Content:
The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.
Thirteen days after the war started, India achieved a clear advantage, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as the new nation of Bangladesh. Approximately 

**VECTOR STORE RETRIEVER**

In [18]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [17]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

previously we might have done:
vec_st = Chroma(
    collection_name="vec_st_retr_collection",
    embedding_function=embedding_model
)

This essentially says:
"Create an empty Chroma vector store."

At this point, Chroma exists, but you haven't necessarily added your documents yet.
Then you could add documents separately:
vec_st.add_documents(documents)

but using from_documents means:
"Create a Chroma vector store AND immediately add these documents to it."

so u don't have to do .add seperately

In [19]:
# Step 2: Initialize embedding model
embedding_model = OpenAIEmbeddings()

# Step 3: Create Chroma vector store in memory since here u won't mention persist_directory->
vec_st = Chroma.from_documents(
    documents = documents, #here itself u pass the documents that u want to add into the vector store
    embedding = embedding_model,
    collection_name="vec_st_retr_collection"
)

In [21]:
# Step 4: Convert vectorstore into a retriever
ret = vec_st.as_retriever(search_kwargs={"k":2})

#as_retriever() is a method of the vector store and it converts vector store into a retriever
#The vector store is responsible for storing/searching vectors.
# The retriever gives you a standardized LangChain interface for retrieving relevant documents.

search_kwargs: "These are additional arguments/options that should be used when the retriever performs its search." and then u r passing {'k':2}

Why is {} used? search_kwargs expects a dictionary
and k means How many documents should the retriever return from the search?

finally, search_kwargs={"k": 2}
→ **Configure the retriever** to return 2 documents

You can potentially have multiple parameters:
search_kwargs={
    "k": 3,
    "filter": {"team": "CSK"}
}
So the dictionary gives you a convenient way of passing multiple search options.

In [22]:
query= "what is Chroma?"
res = ret.invoke(query) #here res is list of doc objects remember i.e what always a result of a retriever

# what happens after doing invoke:
#User query
#     ↓
# Retriever
#     ↓
# Convert query → embedding
#     ↓
# Compare with stored embeddings
#     ↓
# Rank by similarity
#     ↓
# Take top 2 (k=2)
#     ↓
# Return Documents

In [23]:
for i, x in enumerate(res):
  print(f"---result{i+1}---")
  print(x.page_content)

---result1---
Chroma is a vector database optimized for LLM-based search.
---result2---
LangChain helps developers build LLM applications easily.


In [24]:
#now question in mind this perform simkilarity search can be done by vector store itself: imean,
result = vec_st.similarity_search(query, k=2)

for i, x in enumerate(res):
  print(f"---result{i+1}---")
  print(x.page_content)

---result1---
Chroma is a vector database optimized for LLM-based search.
---result2---
LangChain helps developers build LLM applications easily.


You can see that the vector store and the retriever can produce the same results, so why do we need a retriever? The first reason is that retrievers are Runnables, so they can be easily integrated into LangChain chains and pipelines. Also,**a vector store provides its own built-in search methods, whereas a retriever provides a standard retrieval interface through which we can use different and more advanced retrieval strategies.** The retriever shown above is just a vanilla retriever, so it essentially follows the same basic similarity-search strategy provided by the vector store, which is why both can produce the same results. In the upcoming sections, we will see how retrievers can be used to implement more advanced search and retrieval strategies.

In [25]:
# Vector Store
#     ↓
# stores vectors
#     ↓
# provides search methods

# while:
# Retriever
#     ↓
# provides a standard Runnable retrieval interface
#     ↓
# can use different retrieval strategies
#---------------------------------------------------------------------------------
# later u may encounter:
# Vector Store Retriever
#         ↓
#     similarity search

# MMR Retriever
#         ↓
# diversity + relevance

# Self-Query Retriever
#         ↓
# natural-language query
#         ↓
# metadata filtering + similarity search

# Multi-Query Retriever
#         ↓
# one question
#         ↓
# multiple query variations
#         ↓
# retrieve from all

# So the real advantage isn't simply "retriever = better search." The important idea is that retrievers give LangChain a common interface for retrieval, and that interface can represent
# much more sophisticated retrieval strategies than basic vector similarity search.

**MMR**

In [26]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [27]:
from langchain_community.vectorstores import FAISS

In [28]:
embedding_model = OpenAIEmbeddings()

v_st=FAISS.from_documents(
     documents = docs,
     embedding =embedding_model
 )

In [42]:
ret = v_st.as_retriever(
    search_type="mmr",                    # use Maximal Marginal Relevance (MMR) search i mean this parameter specifying as mmr enables MMR
    search_kwargs={"k": 3, "lambda_mult": 1}  # k = number of documents to return; lambda_mult = controls relevance vs diversity(i.e lambda multiplier controls the trade off between those two). range is between 0 to 1 where 1 gives most diverse results and
    #0 behaves as normal similarity search
)

In [43]:
query = "what is langchain?"
res = ret.invoke(query)

In [44]:
for i,x in enumerate(res):
  print(f"---result{i+1}---")
  print(x.page_content)
#change the value of lambda_mult and check the output

---result1---
LangChain is used to build LLM based applications.
---result2---
Embeddings are vector representations of text.
---result3---
MMR helps you get diverse results when doing similarity search.


**MULTI QUERY RETRIEVER**

In [46]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI

In [47]:
from langchain_classic.retrievers import MultiQueryRetriever

why do i ahve to import this retriever? i did not for mmr? can't i just give search type as mqr for this?

You did not need to import anything special for MMR because MMR is a search strategy already supported by the vector store's as_retriever() method.

Vector Store
     ↓
as_retriever()
     ↓
search_type="mmr"
     ↓
MMR search

So mmr is essentially a built-in search type that the vector-store retriever knows how to use.

You cannot do:
ret = v_st.as_retriever(search_type="mqr")

❌ because mqr is not a search type supported by as_retriever().
Instead, MultiQueryRetriever is a separate retriever class.
Then you construct it using another retrieve - this u will seee below:

In [48]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [49]:
# Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings()

# Create FAISS vector store
vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

In [50]:
# Create retrievers - we shall use both normal and mqr retrievers to compare the performance:
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [51]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}), #see, this is a normal retriever too, if u want to perform with mmr u can!
    llm=ChatOpenAI(model="gpt-6-luna")
)
#from_llm() is a class method of MultiQueryRetriever that creates a MultiQueryRetriever by giving it an LLM.
#Think of it as:
#"Create a MultiQueryRetriever and use this LLM to generate multiple versions of the user's query."

In [52]:
#Ambiguious Query
query = "How to improve energy levels and maintain balance?"

In [53]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [54]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)
print("M.Q.R's:")

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

#you can see the query 5 of normal ret output doesn't belong to answer of health and wellness of body


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 4 ---
Deep sleep is crucial for cellular repair and emotional regulation.

--- Result 5 ---
The solar energy system in modern homes helps balance electricity demand.
******************************************************************************************************************************************************
M.Q.R's:

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 3 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improv

**CONTEXTUAL COMPRESSION RETRIEVER**

In [56]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document

In [58]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [59]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [60]:
#create a faiss vector store from the documents:
embedding = OpenAIEmbeddings()
vt_store= FAISS.from_documents(docs, embedding) #or embedding = embedding_model if above "embedding" is named as "embedding_model"

In [61]:
#create a base retriever:
b_r = vt_store.as_retriever(search_kwargs={'k':5})

#set up a compressor using an llm:
x = ChatOpenAI(model = "gpt-6-luna")
b_c = LLMChainExtractor.from_llm(x) #or llm == llm if x is named by "llm" instead

#create conceptual compression retriever:
compression_retriever = ContextualCompressionRetriever(
    base_retriever= b_r,
    base_compressor = b_c
)

In [64]:
query = "what is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [65]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.

--- Result 3 ---
Photosynthesis does not occur in animal cells.
